# Introduction to YOLO: Real-Time Object Detection 🎯

Welcome! In this tutorial, you'll learn how to use **YOLO (You Only Look Once)** - one of the most popular and powerful object detection models in the world.

By the end of this notebook, you'll be able to:
- ✅ Understand how YOLO works (at a high level)
- ✅ Set up and run YOLO on your own images
- ✅ Use different YOLO features (detection, segmentation, pose estimation)
- ✅ Fine-tune YOLO on your own custom dataset

---

## 📚 Important: Documentation & Learning to Read Docs

Before we dive in, let's talk about something crucial for your AI journey:

**Official Documentation:**
- 🔗 [Ultralytics YOLO Docs](https://docs.ultralytics.com/) - The official documentation
- 🔗 [YOLO GitHub Repository](https://github.com/ultralytics/ultralytics)
- 🔗 [Ultralytics HUB](https://hub.ultralytics.com/) - No-code training platform

### Why Reading Documentation Matters

As you advance in AI/ML, you'll constantly use external libraries and pre-trained models. **No tutorial can cover everything** - documentation is your primary resource.

**Tips for reading docs effectively:**
1. **Start with "Getting Started" or "Quickstart"** - Get something working first
2. **Skim the API reference** - Know what's available, even if you don't memorize it
3. **Search for examples** - Most docs have code examples you can copy and modify
4. **Don't read everything** - Look up what you need, when you need it

The Ultralytics docs are excellent - well-organized with lots of examples. Use them!


---

## 🧠 How Does YOLO Work? (Technical Overview)

### The Problem: Object Detection

**Object detection** is harder than image classification:
- Classification: "Is there a cat in this image?" → Yes/No
- Detection: "Where are ALL the objects, and what are they?" → Multiple bounding boxes + labels

### Traditional Approach (Before YOLO)

Older methods used a **two-stage approach**:
1. **Stage 1:** Generate thousands of "region proposals" (possible object locations)
2. **Stage 2:** Classify each region individually

This was **slow** - you had to run the classifier thousands of times per image!

### YOLO's Innovation: "You Only Look Once"

YOLO revolutionized object detection by doing everything in **one pass**:

1. **Divide the image into a grid** (e.g., 13×13 cells)
2. **Each cell predicts:**
   - Multiple bounding boxes (location + size)
   - Confidence score (how sure it is there's an object)
   - Class probabilities (what type of object)
3. **All predictions happen simultaneously** in one forward pass through the network

### Why YOLO is Fast

| Method | Speed | Why |
|--------|-------|-----|
| Two-stage (R-CNN) | ~2-5 FPS | Runs classifier thousands of times |
| **YOLO** | **30-150+ FPS** | Single neural network pass |

This makes YOLO perfect for **real-time applications** like:
- Self-driving cars
- Security cameras
- Sports analysis
- Robotics

### YOLO Architecture (Simplified)

```
Input Image (640×640)
        ↓
   [Backbone CNN]        ← Extracts features (patterns, edges, shapes)
        ↓
   [Neck/FPN]            ← Combines features at different scales
        ↓
   [Detection Head]      ← Outputs boxes, confidence, class predictions
        ↓
   Predictions (boxes + labels)
```

**Key insight:** YOLO treats detection as a **regression problem** (predicting numbers), not a classification problem. This is why it can be so fast!


---

# Part 1: Setting Up YOLO 🔧

We'll use **Ultralytics YOLOv8** - the latest and most user-friendly version of YOLO.

YOLOv8 is incredibly easy to use - just a few lines of code!


In [ ]:
# Step 1: Install the ultralytics package
# This includes YOLOv8 and all its dependencies
# Run this cell once - it may take a minute

%pip install ultralytics -q

# Verify installation
import ultralytics
ultralytics.checks()

print("\n✅ YOLO is ready to use!")


In [ ]:
# Step 2: Import the libraries we'll need
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# For displaying images nicely in the notebook
%matplotlib inline

print("✅ All imports successful!")


---

# Part 2: Using the Pre-trained YOLO Model 🎯

YOLO comes with **pre-trained weights** - it's already learned to detect 80 common objects from the COCO dataset!

## Available Model Sizes

| Model | Size | Speed | Accuracy | Use Case |
|-------|------|-------|----------|----------|
| YOLOv8n | Nano | ⚡ Fastest | Lower | Mobile, edge devices |
| YOLOv8s | Small | Fast | Good | Balanced performance |
| YOLOv8m | Medium | Medium | Better | General use |
| YOLOv8l | Large | Slower | High | When accuracy matters |
| YOLOv8x | XLarge | Slowest | Highest | Maximum accuracy |

**Tip:** Start with `yolov8n` or `yolov8s` for learning - they're faster to download and run!


In [ ]:
# Load a pre-trained YOLOv8 model
# The first time you run this, it will download the model weights (~6MB for nano)

model = YOLO('yolov8n.pt')  # 'n' = nano (smallest, fastest)

print("✅ Model loaded!")
print(f"Model type: {model.task}")  # 'detect' for object detection
print(f"Number of classes: {len(model.names)}")  # 80 classes in COCO dataset


In [ ]:
# Let's see what objects YOLO can detect out-of-the-box!
# These are the 80 COCO dataset classes

print("🏷️ YOLO can detect these 80 object types:\n")

# Display in a nice grid format
class_names = list(model.names.values())
for i in range(0, len(class_names), 5):
    row = class_names[i:i+5]
    print("  " + " | ".join(f"{name:15}" for name in row))


## 2.1 Running Detection on an Image

Let's run YOLO on a sample image! We'll download one from the internet.


In [ ]:
# Download a sample image from the internet
# This is a busy street scene with cars, people, etc.

image_url = "https://ultralytics.com/images/bus.jpg"

# Download and save the image
response = requests.get(image_url)
img = Image.open(BytesIO(response.content))
img.save("sample_image.jpg")

# Display the original image
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.title("Original Image (before detection)", fontsize=14)
plt.axis('off')
plt.show()

print(f"Image size: {img.size[0]} x {img.size[1]} pixels")


In [ ]:
# Run object detection! 🎯
# This is the magic line - YOLO will find all objects in the image

results = model("sample_image.jpg")

# The results object contains all the detections
# Let's see what YOLO found!
print("🔍 Detection Results:")
print(f"   Number of objects detected: {len(results[0].boxes)}")
print(f"   Processing time: {results[0].speed['inference']:.1f}ms")


In [ ]:
# Display the image with bounding boxes drawn on it
# YOLO provides a convenient plot() method

plt.figure(figsize=(12, 8))
result_image = results[0].plot()  # This draws boxes on the image
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.title("YOLO Detection Results - Objects Found!", fontsize=14)
plt.axis('off')
plt.show()


In [ ]:
# Let's look at the details of each detected object
# This is useful for understanding what YOLO outputs

print("📊 Detailed Detection Results:\n")
print(f"{'Object':<15} {'Confidence':<12} {'Bounding Box (x1, y1, x2, y2)'}")
print("-" * 60)

# Loop through each detected object
for box in results[0].boxes:
    # Get the class name
    class_id = int(box.cls[0])
    class_name = model.names[class_id]
    
    # Get the confidence score (0-1, higher = more confident)
    confidence = float(box.conf[0])
    
    # Get the bounding box coordinates
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    
    print(f"{class_name:<15} {confidence:.1%}        ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")

print("\n💡 What these numbers mean:")
print("   • Confidence: How sure YOLO is (90% = very confident)")
print("   • Bounding Box: Top-left (x1,y1) and bottom-right (x2,y2) corners")


---

# Part 3: YOLO Features & Capabilities 🛠️

YOLOv8 isn't just for detecting objects - it can do several different tasks!

## Available Tasks

| Task | Model | What it does |
|------|-------|--------------|
| **Detection** | yolov8n.pt | Finds objects with bounding boxes |
| **Segmentation** | yolov8n-seg.pt | Finds objects with pixel-perfect masks |
| **Pose Estimation** | yolov8n-pose.pt | Detects human body keypoints |
| **Classification** | yolov8n-cls.pt | Classifies entire images |
| **Oriented Detection** | yolov8n-obb.pt | Detects rotated objects |

Let's try some of these!


## 3.1 Instance Segmentation

Segmentation gives you **pixel-perfect masks** instead of just boxes. This is useful when you need to know the exact shape of objects.


In [ ]:
# Load the segmentation model (adds "-seg" to the model name)
seg_model = YOLO('yolov8n-seg.pt')

# Run segmentation on the same image
seg_results = seg_model("sample_image.jpg")

# Display results
plt.figure(figsize=(12, 8))
seg_image = seg_results[0].plot()
plt.imshow(cv2.cvtColor(seg_image, cv2.COLOR_BGR2RGB))
plt.title("Instance Segmentation - Pixel-Perfect Object Masks!", fontsize=14)
plt.axis('off')
plt.show()

print("💡 Notice how each object now has a colored MASK, not just a box!")
print("   This is useful for: background removal, object counting, area measurement")


## 3.2 Pose Estimation

Pose estimation finds **human body keypoints** - useful for fitness apps, sports analysis, and more!


In [ ]:
# Download an image with people for pose estimation
pose_url = "https://ultralytics.com/images/bus.jpg"  # Has people in it

# Load the pose estimation model
pose_model = YOLO('yolov8n-pose.pt')

# Run pose estimation
pose_results = pose_model("sample_image.jpg")

# Display results
plt.figure(figsize=(12, 8))
pose_image = pose_results[0].plot()
plt.imshow(cv2.cvtColor(pose_image, cv2.COLOR_BGR2RGB))
plt.title("Pose Estimation - Human Body Keypoints!", fontsize=14)
plt.axis('off')
plt.show()

print("💡 YOLO detected body keypoints: shoulders, elbows, wrists, hips, knees, ankles")
print("   Use cases: fitness apps, sports analysis, gesture recognition, AR/VR")


## 3.3 Useful Detection Parameters

YOLO has several parameters you can adjust:


In [ ]:
# Example: Adjusting detection parameters

# Run with custom settings
results = model(
    "sample_image.jpg",
    conf=0.5,          # Minimum confidence threshold (0-1). Higher = fewer but more confident detections
    iou=0.7,           # IoU threshold for NMS (non-max suppression). Higher = allows more overlapping boxes
    classes=[0, 2],    # Only detect specific classes (0=person, 2=car). None = all classes
    verbose=False      # Don't print output to console
)

# Show results
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
plt.title("Filtered Detection: Only People (0) and Cars (2), Confidence > 50%", fontsize=12)
plt.axis('off')
plt.show()

print("\n📋 Common Parameters:")
print("   • conf: Minimum confidence (default 0.25). Increase to reduce false positives")
print("   • iou: IoU threshold (default 0.7). Affects overlapping detections")
print("   • classes: List of class IDs to detect (None = all)")
print("   • imgsz: Input image size (default 640). Larger = slower but more accurate")
print("   • device: 'cpu' or 'cuda' (GPU) for processing")


---

# Part 4: Fine-Tuning YOLO on Custom Data 🎓

The pre-trained YOLO can detect 80 objects, but what if you want to detect something specific like:
- Different types of birds
- Manufacturing defects
- Medical images
- Your company's products

You need to **fine-tune** (also called transfer learning) the model on your own data!

## The Problem: YOLO Doesn't Know Everything

Let's demonstrate this with a real example. We'll try to detect **African wildlife** - specifically buffalo, elephant, rhino, and zebra. 

While YOLO knows "elephant" and "zebra" from COCO, it has **never seen "buffalo" or "rhino"** - these aren't in the 80 COCO classes!

Let's see what happens when we try...


## 4.1 BEFORE Fine-Tuning: Watch YOLO Fail! 😅

First, let's download some African wildlife images and see how the pre-trained YOLO performs.


In [ ]:
# First, let's download the African Wildlife dataset
# This will give us images of buffalo, elephant, rhino, and zebra to test with!

from ultralytics.data.utils import check_det_dataset

print("📥 Downloading African Wildlife dataset...")
print("   (This contains buffalo, elephant, rhino, zebra images)\n")

# This downloads the dataset and returns info about it
dataset_info = check_det_dataset('african-wildlife.yaml')

print("\n✅ Dataset downloaded!")
print(f"   Location: {dataset_info.get('path', 'datasets/african-wildlife')}")


In [ ]:
# Let's grab 3 distinct animals from the validation set
import os
import glob

# Get all validation images
all_val_images = glob.glob('datasets/african-wildlife/images/val/*.jpg')

# Images are named like "1 (xxx).jpg", "2 (xxx).jpg", etc.
# where 1=buffalo, 2=elephant, 3=rhino, 4=zebra (based on typical naming)
# Let's get one image from each distinct prefix

test_images = []
seen_prefixes = set()


for img_path in sorted(all_val_images):
    filename = os.path.basename(img_path)
    # Get the prefix (first character/number before space or parenthesis)
    prefix = filename.split()[0] if ' ' in filename else filename[0]
    
    # Skip elephant (2) and zebra (4) since they're already in COCO
    # Only use buffalo (1) and rhino (3) - animals NOT in COCO!
    if prefix in ['1', '3'] and prefix not in seen_prefixes and len(test_images) < 2:
        test_images.append(img_path)
        seen_prefixes.add(prefix)

print(f"✅ Selected {len(test_images)} images (buffalo & rhino - NOT in COCO):")
for img in test_images:
    print(f"   • {os.path.basename(img)}")


In [ ]:
# Load the pre-trained model and see what it knows
pretrained_model = YOLO('yolov8n.pt')

# Show what COCO classes are related to animals
print("🐾 Animal classes in COCO (what pre-trained YOLO knows):")
animal_classes = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']
for cls in animal_classes:
    class_id = [k for k, v in pretrained_model.names.items() if v == cls]
    if class_id:
        print(f"   ✓ {cls}")

print("\n❌ NOT in COCO: buffalo, rhino (our dataset has these!)")
print("   The pre-trained model has never seen these animals!")

# Run detection on our wildlife test images
if test_images:
    num_images = min(3, len(test_images))
    fig, axes = plt.subplots(1, num_images, figsize=(5*num_images, 5))
    if num_images == 1:
        axes = [axes]
    
    print(f"\n🔍 Running pre-trained YOLO on {num_images} wildlife images...")
    
    for idx, img_path in enumerate(test_images[:num_images]):
        results = pretrained_model(img_path, verbose=False)
        result_img = results[0].plot()
        axes[idx].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
        
        # What did YOLO detect?
        detections = []
        for box in results[0].boxes:
            class_name = pretrained_model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            detections.append(f"{class_name} ({conf:.0%})")
        
        detected_str = ", ".join(detections) if detections else "Nothing!"
        axes[idx].set_title(f'Pre-trained detected:\n{detected_str}', fontsize=10, color='red')
        axes[idx].axis('off')
    
    plt.suptitle('🚫 BEFORE Fine-Tuning: Pre-trained YOLO on African Wildlife', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n😱 OBSERVATION:")
    print("   The pre-trained YOLO likely:")
    print("   • Detects NOTHING (doesn't recognize buffalo/rhino)")
    print("   • Or MISCLASSIFIES as cow, bear, elephant, etc.")
    print("\n   We need to FINE-TUNE the model on this wildlife data!")
else:
    print("\n⚠️ No test images found. Run the dataset download cell first.")


## 4.2 Fine-Tuning: Teaching YOLO New Tricks 🎓

Now let's fine-tune YOLO on the **African Wildlife dataset** so it can recognize buffalo, elephant, rhino, and zebra!

### How Fine-Tuning Works

1. **Start with pre-trained weights** - YOLO already knows how to detect objects in general
2. **Show it YOUR images** - With your custom labels
3. **Let it learn** - The model adjusts to your specific objects
4. **Result** - A custom model that detects YOUR objects!

This is much faster than training from scratch because YOLO already understands images (edges, shapes, textures).


In [ ]:
# Fine-tune YOLO on the African Wildlife dataset!
# This dataset is built into Ultralytics and will auto-download

# Load a fresh model starting from pre-trained COCO weights
model_to_train = YOLO('yolov8n.pt')

print("🎓 Starting fine-tuning on African Wildlife dataset...")
print("   Classes: buffalo, elephant, rhino, zebra")
print("   This will take a few minutes...\n")

# Train the model
# NOTE: We use few epochs for demo. Real training would use 50-100+ epochs
train_results = model_to_train.train(
    data='african-wildlife.yaml',  # Built-in dataset (auto-downloads)
    epochs=3,                      # More epochs = better results (use 50+ for real projects)
    imgsz=640,                      # Image size
    batch=8,                        # Batch size (reduce if out of memory)
    patience=5,                     # Early stopping
    name='wildlife_model',          # Name for this training run
    verbose=True
)

print("\n✅ Fine-tuning complete!")


## 4.3 AFTER Fine-Tuning: The Moment of Truth! 🎉

Now let's load our fine-tuned model and test it on the same wildlife images!


In [ ]:
# Load our fine-tuned model
import glob

# Find the trained model (path may vary based on run number)
model_paths = glob.glob('runs/detect/wildlife_model*/weights/best.pt')
if not model_paths:
    model_paths = glob.glob('runs/detect/train*/weights/best.pt')

if model_paths:
    best_model_path = sorted(model_paths)[-1]  # Get most recent
    finetuned_model = YOLO(best_model_path)
    print(f"✅ Fine-tuned model loaded from: {best_model_path}")
    print(f"\n🏷️ This model knows these classes:")
    for class_id, class_name in finetuned_model.names.items():
        print(f"   {class_id}: {class_name}")
else:
    print("⚠️ No trained model found. Please run the training cell first!")
    finetuned_model = None


In [ ]:
# Now the exciting part - BEFORE vs AFTER comparison! 🎬

if finetuned_model is None:
    print("⚠️ Please run the training and model loading cells first!")
elif not test_images:
    print("⚠️ No test images available. Please run the dataset download cell first!")
else:
    num_images = min(2, len(test_images))
    fig, axes = plt.subplots(2, num_images, figsize=(8*num_images, 12))
    
    for col, img_path in enumerate(test_images[:num_images]):
        # ===== TOP ROW: BEFORE (Pre-trained model) =====
        results_before = pretrained_model(img_path, verbose=False)
        result_img_before = results_before[0].plot()
        axes[0, col].imshow(cv2.cvtColor(result_img_before, cv2.COLOR_BGR2RGB))
        
        # What did pre-trained model detect?
        detections_before = []
        for box in results_before[0].boxes:
            class_name = pretrained_model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            detections_before.append(f"{class_name} ({conf:.0%})")
        detected_str_before = ", ".join(detections_before) if detections_before else "Nothing!"
        
        axes[0, col].set_title(f'🚫 BEFORE (Pre-trained)\nDetected: {detected_str_before}', 
                              fontsize=12, color='red')
        axes[0, col].axis('off')
        
        # ===== BOTTOM ROW: AFTER (Fine-tuned model) =====
        results_after = finetuned_model(img_path, verbose=False)
        result_img_after = results_after[0].plot()
        axes[1, col].imshow(cv2.cvtColor(result_img_after, cv2.COLOR_BGR2RGB))
        
        # What did fine-tuned model detect?
        detections_after = []
        for box in results_after[0].boxes:
            class_name = finetuned_model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            detections_after.append(f"{class_name} ({conf:.0%})")
        detected_str_after = ", ".join(detections_after) if detections_after else "Nothing!"
        
        axes[1, col].set_title(f'✅ AFTER (Fine-tuned)\nDetected: {detected_str_after}', 
                              fontsize=12, color='green')
        axes[1, col].axis('off')
    
    plt.suptitle('🎯 BEFORE vs AFTER Fine-Tuning Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n🎉 SUCCESS! After fine-tuning:")
    print("   • The model now correctly identifies buffalo, elephant, rhino, and zebra!")
    print("   • It learned these new classes in just a few minutes of training")
    print("   • The pre-trained features (edges, shapes) helped it learn quickly")
    print("\n💡 KEY TAKEAWAY:")
    print("   Fine-tuning lets you customize YOLO for YOUR specific use case!")


## 4.4 Creating Your Own Custom Dataset

Now that you've seen fine-tuning in action, here's how to create your own dataset!

### Dataset Structure

YOLO expects your data in a specific format:

```
dataset/
├── train/
│   ├── images/          # Training images
│   │   ├── img1.jpg
│   │   ├── img2.jpg
│   │   └── ...
│   └── labels/          # Labels (same name as images, .txt extension)
│       ├── img1.txt
│       ├── img2.txt
│       └── ...
├── val/
│   ├── images/          # Validation images
│   └── labels/          # Validation labels
└── data.yaml            # Dataset configuration file
```

### Label Format

Each `.txt` label file contains one line per object:
```
class_id center_x center_y width height
```

All values are **normalized** (0-1 relative to image size):
- `class_id`: Integer class number (0, 1, 2, ...)
- `center_x`: X coordinate of box center (0-1)
- `center_y`: Y coordinate of box center (0-1)
- `width`: Box width (0-1)
- `height`: Box height (0-1)


In [ ]:
# Example: What a data.yaml file looks like
# This is the configuration file that tells YOLO about your dataset

example_yaml = """
# data.yaml - Dataset configuration

# Paths (can be absolute or relative)
path: /path/to/your/dataset    # Root directory
train: train/images            # Training images folder (relative to path)
val: val/images                # Validation images folder

# Class names - list your object types here
names:
  0: cat
  1: dog
  2: bird

# Number of classes
nc: 3
"""

print("📄 Example data.yaml file:")
print(example_yaml)

print("\n💡 TIPS for creating your own dataset:")
print("   1. 🏷️ Use labeling tools like:")
print("      • Roboflow (free tier, web-based, exports in YOLO format)")
print("      • LabelImg (free, open-source, desktop app)")
print("      • CVAT (free, open-source, web-based)")
print("\n   2. 📊 Dataset size guidelines:")
print("      • Minimum: ~50-100 images per class")
print("      • Recommended: 500+ images per class")
print("      • More data = better results!")
print("\n   3. 📸 Include variety:")
print("      • Different angles and distances")
print("      • Various lighting conditions")
print("      • Different backgrounds")
print("\n   4. 📂 Split your data:")
print("      • ~80% for training")
print("      • ~20% for validation")


---

# Summary & Next Steps 🎓

## What You Learned

1. **How YOLO works** - Single-pass detection using a grid system (fast!)
2. **Using pre-trained models** - Load and run detection in just 3 lines of code
3. **Different capabilities** - Detection, segmentation, pose estimation
4. **Customization** - Parameters for filtering and tuning results
5. **Fine-tuning** - Train YOLO on your own custom data
6. **The power of transfer learning** - Pre-trained models can quickly learn new objects!

## Key Takeaway: Before vs After Fine-Tuning

| Scenario | Pre-trained YOLO | Fine-tuned YOLO |
|----------|------------------|-----------------|
| Buffalo image | ❌ "cow" or nothing | ✅ "buffalo" |
| Rhino image | ❌ "elephant" or nothing | ✅ "rhino" |

**Fine-tuning lets you customize YOLO for YOUR specific use case!**

## Quick Reference

```python
# Load models
model = YOLO('yolov8n.pt')           # Detection
model = YOLO('yolov8n-seg.pt')       # Segmentation  
model = YOLO('yolov8n-pose.pt')      # Pose estimation

# Run inference
results = model('image.jpg')
results = model('image.jpg', conf=0.5, classes=[0, 1])

# Fine-tune on custom data
model.train(data='your_data.yaml', epochs=50)

# Load your fine-tuned model
custom_model = YOLO('runs/detect/train/weights/best.pt')
```

## Explore More

- 🎥 **Video processing**: `model('video.mp4')`
- 📹 **Webcam**: `model(source=0)`
- 🔄 **Object tracking**: `model.track('video.mp4')`
- ☁️ **Ultralytics HUB**: No-code training platform

## Resources

- [Official Documentation](https://docs.ultralytics.com/)
- [GitHub Repository](https://github.com/ultralytics/ultralytics)
- [Roboflow](https://roboflow.com/) - Dataset creation and management
- [Ultralytics Discord](https://discord.gg/ultralytics) - Community support

---

**Great job!** You now have the foundations to use YOLO for object detection in your own projects! 🚀

Remember: The documentation is your friend. When you get stuck, check the docs first!
